In [1]:
from pathlib import Path
import re
import unicodedata
from zoneinfo import ZoneInfo

import json
import pandas as pd
import soccerdata as sd

players = [
    "Vanja Milinković-Savić",
    "Alex Meret",
    "Luka Vuskovic",
    "Jeff Chabot",
    "Mario Hermoso",
    "Lewis Dunk",
    "Jérémy Jacquet",
    "Kristoffer Ajer",
    "Matija Katic",
    "Scott McTominay",
    "Can Uzun",
    "Pierre-Emile Højbjerg",
    "Edu Exposito",
    "Pape Gueye",
    "Erling Haaland",
    "Igor Thiago",
    "Mattia Zaccagni",
    "Randal Kolo Muani",
    "Alexander Isak",
    "Dominic Calvert-Lewin",
]


def normalize_player_name(name):
    name = re.sub(r"^\s*\d+\s*", "", str(name or ""))
    name = " ".join(name.replace("\u00a0", " ").split()).casefold()
    return "".join(
        char
        for char in unicodedata.normalize("NFKD", name)
        if not unicodedata.combining(char)
    )


workbook_path = Path("data/fantasy_data.xlsx")
workbook = pd.read_excel(workbook_path, sheet_name=None, usecols=["Player", "Club"])
team_by_player = {}
for sheet_name, sheet in workbook.items():
    season = int(sheet_name.rsplit(" ", maxsplit=1)[-1])
    for player_name, team in sheet[["Player", "Club"]].itertuples(index=False, name=None):
        key = normalize_player_name(player_name)
        if key and (key not in team_by_player or season > team_by_player[key][0]):
            team_by_player[key] = (season, team)

my_team = pd.DataFrame(
    {
        "player_name": players,
        "team": [
            team_by_player.get(normalize_player_name(player), (None, None))[1]
            for player in players
        ],
    }
)
LEAGUE_BY_TEAM = {
    "NAP": "ITA-Serie A", "LAZ": "ITA-Serie A", "JUV": "ITA-Serie A", "ROM": "ITA-Serie A",
    "VFB": "GER-Bundesliga", "SGE": "GER-Bundesliga", "OM": "FRA-Ligue 1",
    "ESY": "ESP-La Liga", "VIL": "ESP-La Liga",
    "BHA": "ENG-Premier League", "LIV": "ENG-Premier League",
    "BRE": "ENG-Premier League", "MCI": "ENG-Premier League", "LEE": "ENG-Premier League",
}
TEAM_ALIASES = {
    "NAP": ("Napoli", "Naples"), "BHA": ("Brighton and Hove Albion", "Brighton & Hove Albion", "Brighton"),
    "VFB": ("Stuttgart", "VfB Stuttgart"), "ROM": ("Roma", "AS Roma"),
    "LIV": ("Liverpool",), "BRE": ("Brentford",),
    "SGE": ("Eintracht Frankfurt", "Frankfurt"), "OM": ("Marseille", "Olympique de Marseille"),
    "ESY": ("Espanyol", "RCD Espanyol", "Espanyol Barcelona"),
    "VIL": ("Villarreal", "Villarreal CF"), "MCI": ("Manchester City",),
    "LAZ": ("Lazio",), "JUV": ("Juventus",), "LEE": ("Leeds United", "Leeds"),
    "RBLEIPZIG": ("RB Leipzig", "RasenBallsport Leipzig"),
    "PADERBORN": ("Paderborn", "SC Paderborn", "SC Paderborn 07"),
}

def normalize_team_name(name):
    value = unicodedata.normalize("NFKD", str(name or "")).casefold()
    return "".join(char for char in value if char.isalnum())

def is_team(name, club_code):
    return normalize_team_name(name) in {
        normalize_team_name(alias) for alias in TEAM_ALIASES.get(club_code, (club_code,))
    }

def canonical_team_key(name):
    normalized = normalize_team_name(name)
    for code, aliases in TEAM_ALIASES.items():
        if normalized == normalize_team_name(code) or normalized in {
            normalize_team_name(alias) for alias in aliases
        }:
            return code
    return normalized

team_contexts = {
    (LEAGUE_BY_TEAM[team], team)
    for team in my_team["team"].dropna().unique()
    if team in LEAGUE_BY_TEAM
}
leagues = sorted({league for league, _ in team_contexts})
espn = sd.ESPN(leagues=leagues, seasons=2026)
whoscored = sd.WhoScored(leagues=leagues, seasons=2026, headless=True)
refresh_schedules = False
now_utc = pd.Timestamp.now(tz="UTC")
espn_schedule = espn.read_schedule(force_cache=not refresh_schedules).reset_index()
espn_schedule["date_utc"] = pd.to_datetime(espn_schedule["date"], utc=True, errors="coerce")
whoscored_schedule = whoscored.read_schedule(
    force_cache=not refresh_schedules
).reset_index()
whoscored_schedule["date_utc"] = pd.to_datetime(
    whoscored_schedule["date"], utc=True, errors="coerce"
)
understat = sd.Understat(leagues=leagues, seasons=2026)
understat_matches = understat.read_team_match_stats(force_cache=True).reset_index()
understat_matches["date_utc"] = pd.to_datetime(
    understat_matches["date"], utc=True, errors="coerce"
)
understat_matches = understat_matches[understat_matches["date_utc"] < now_utc].copy()
home_xg_rows = pd.DataFrame({
    "league": understat_matches["league"],
    "team_key": understat_matches["home_team"].map(canonical_team_key),
    "date_utc": understat_matches["date_utc"],
    "xg": pd.to_numeric(understat_matches["home_xg"], errors="coerce"),
})
away_xg_rows = pd.DataFrame({
    "league": understat_matches["league"],
    "team_key": understat_matches["away_team"].map(canonical_team_key),
    "date_utc": understat_matches["date_utc"],
    "xg": pd.to_numeric(understat_matches["away_xg"], errors="coerce"),
})
home_xg_rows["venue"] = "Home"
away_xg_rows["venue"] = "Away"
team_match_xg = pd.concat([home_xg_rows, away_xg_rows], ignore_index=True)
team_match_xg = team_match_xg.dropna(subset=["xg"])
venue_xg = (
    team_match_xg.groupby(["league", "team_key", "venue"], as_index=False)["xg"]
    .mean()
)
venue_xg_lookup = {
    (row.league, row.team_key, row.venue): row.xg
    for row in venue_xg.itertuples(index=False)
}
venue_xg["league_percentile"] = venue_xg.groupby(["league", "venue"])["xg"].rank(pct=True)
difficulty_labels = ["very easy", "easy", "average", "hard", "very hard"]
def difficulty_from_percentile(percentile):
    if pd.isna(percentile):
        return None
    if percentile <= 0.2:
        return difficulty_labels[0]
    if percentile <= 0.4:
        return difficulty_labels[1]
    if percentile <= 0.6:
        return difficulty_labels[2]
    if percentile <= 0.8:
        return difficulty_labels[3]
    return difficulty_labels[4]
venue_xg["difficulty"] = venue_xg["league_percentile"].map(difficulty_from_percentile)
difficulty_lookup = {
    (row.league, row.team_key, row.venue): row.difficulty
    for row in venue_xg.itertuples(index=False)
}

fixture_plans = {}
lookup_issues = []
for league, club_code in team_contexts:
    league_schedule = espn_schedule[espn_schedule["league"] == league].copy()
    club_fixtures = league_schedule[
        league_schedule["home_team"].map(lambda name: is_team(name, club_code))
        | league_schedule["away_team"].map(lambda name: is_team(name, club_code))
    ]
    upcoming = club_fixtures[club_fixtures["date_utc"] >= now_utc].sort_values("date_utc")
    if upcoming.empty:
        lookup_issues.append(f"No upcoming fixture found for {club_code} ({league}).")
        continue
    next_match = upcoming.iloc[0]
    is_home = is_team(next_match["home_team"], club_code)
    team_full_name = next_match["home_team"] if is_home else next_match["away_team"]
    opponent = next_match["away_team"] if is_home else next_match["home_team"]
    venue = "Home" if is_home else "Away"
    recent_matches = club_fixtures[
        club_fixtures["date_utc"] < next_match["date_utc"]
    ].sort_values("date_utc", ascending=False).head(4)
    fixture_plans[(league, club_code)] = {
        "opponent": opponent, "team_full_name": team_full_name,
        "next_date": next_match["date_utc"], "venue": venue,
        "recent_matches": recent_matches,
    }
    if len(recent_matches) < 4:
        lookup_issues.append(f"{club_code} ({league}): only {len(recent_matches)} recent match(es) available.")

recent_match_ids = list(dict.fromkeys(
    int(match_id) for plan in fixture_plans.values()
    for match_id in plan["recent_matches"]["game_id"].dropna()
))

def read_recent_lineups(match_ids):
    matches = espn_schedule[espn_schedule["game_id"].isin(match_ids)]
    rows = []
    for match in matches.itertuples(index=False):
        game_id = int(match.game_id)
        url = f"http://site.api.espn.com/apis/site/v2/sports/soccer/{match.league_id}/summary?event={game_id}"
        summary = json.load(espn.get(url, espn.data_dir / f"Summary_{game_id}.json"))
        rosters = summary.get("rosters", [])
        if not rosters:
            lookup_issues.append(f"ESPN lineup unavailable for match {game_id}.")
            continue
        for roster_group in rosters:
            team_name = roster_group.get("team", {}).get("displayName")
            for player in roster_group.get("roster", []):
                subbed_in = player.get("subbedIn", False)
                entered = subbed_in.get("didSub", False) if isinstance(subbed_in, dict) else bool(subbed_in)
                rows.append({
                    "league": match.league, "game": match.game, "team": team_name,
                    "player": player["athlete"]["displayName"],
                    "played": bool(player.get("starter", False)) or entered,
                })
    return pd.DataFrame(rows)

lineups = read_recent_lineups(recent_match_ids) if recent_match_ids else pd.DataFrame()
next_whoscored_ids = []
for (league, club_code), plan in fixture_plans.items():
    schedule = whoscored_schedule[whoscored_schedule["league"] == league].copy()
    candidates = schedule[
        schedule["home_team"].map(lambda name: is_team(name, club_code))
        | schedule["away_team"].map(lambda name: is_team(name, club_code))
    ].copy()
    candidates["date_gap"] = (candidates["date_utc"] - plan["next_date"]).abs()
    candidates = candidates[candidates["date_gap"] <= pd.Timedelta(days=3)].sort_values("date_gap")
    if candidates.empty:
        lookup_issues.append(f"WhoScored fixture not matched for {club_code} ({league}).")
    else:
        next_whoscored_ids.append(int(candidates.iloc[0]["game_id"]))

missing_players = (
    whoscored.read_missing_players(match_id=list(dict.fromkeys(next_whoscored_ids)), force_cache=True)
    if next_whoscored_ids else pd.DataFrame()
)
missing_player_keys = set()
if not missing_players.empty:
    for row in missing_players.reset_index().itertuples(index=False):
        for league, club_code in team_contexts:
            if row.league == league and is_team(row.team, club_code):
                missing_player_keys.add((league, club_code, normalize_player_name(row.player)))

def recent_play_probability(player_name, league, club_code):
    plan = fixture_plans.get((league, club_code))
    if plan is None or lineups.empty:
        return None
    key = normalize_player_name(player_name)
    played_count = observed_count = 0
    for match in plan["recent_matches"].itertuples(index=False):
        rows = lineups[
            (lineups["league"] == league) & (lineups["game"] == match.game)
            & lineups["team"].map(lambda name: is_team(name, club_code))
        ]
        if rows.empty:
            continue
        observed_count += 1
        appeared = rows[rows["player"].map(normalize_player_name) == key]["played"]
        if not appeared.empty and appeared.any():
            played_count += 1
    if observed_count == 0:
        return None
    if (league, club_code, key) in missing_player_keys:
        return 0.0
    return played_count / observed_count

my_team_enriched = my_team.copy()
my_team_enriched["opponent"] = None
my_team_enriched["team_full_name"] = None
my_team_enriched["fixture_time_utc"] = None
my_team_enriched["venue"] = None
my_team_enriched["team_xg"] = None
my_team_enriched["opponent_xg"] = None
my_team_enriched["opposition_difficulty"] = None
my_team_enriched["expected_to_play"] = None
for row_index, player in my_team_enriched.iterrows():
    club_code = player["team"]
    league = LEAGUE_BY_TEAM.get(club_code)
    if league is None:
        lookup_issues.append(f"No current team/league mapping found for {player['player_name']}.")
        continue
    plan = fixture_plans.get((league, club_code))
    if plan is not None:
        my_team_enriched.at[row_index, "opponent"] = plan["opponent"]
        my_team_enriched.at[row_index, "team_full_name"] = plan["team_full_name"]
        my_team_enriched.at[row_index, "fixture_time_utc"] = (
            plan["next_date"].tz_convert(ZoneInfo("Europe/Rome")).strftime("%Y-%m-%d %H:%M %Z")
        )
        my_team_enriched.at[row_index, "venue"] = plan["venue"]
        team_key = canonical_team_key(plan["team_full_name"])
        opponent_key = canonical_team_key(plan["opponent"])
        team_venue = plan["venue"]
        opponent_venue = "Away" if team_venue == "Home" else "Home"
        my_team_enriched.at[row_index, "team_xg"] = venue_xg_lookup.get(
            (league, team_key, team_venue)
        )
        opponent_xg = venue_xg_lookup.get((league, opponent_key, opponent_venue))
        my_team_enriched.at[row_index, "opponent_xg"] = opponent_xg
        my_team_enriched.at[row_index, "opposition_difficulty"] = difficulty_lookup.get(
            (league, opponent_key, opponent_venue)
        )
        my_team_enriched.at[row_index, "expected_to_play"] = recent_play_probability(
            player["player_name"], league, club_code
        )

display_table = my_team_enriched.rename(columns={
    "player_name": "Player", "team_full_name": "Team",
    "fixture_time_utc": "Fixture date/time (Rome)", "venue": "Home/Away",
    "team_xg": "Team xG", "opponent_xg": "Opponent xG",
    "opposition_difficulty": "Opposition difficulty",
    "expected_to_play": "Expected to play", "opponent": "Next opponent",
})[["Player", "Team", "Fixture date/time (Rome)", "Home/Away", "Team xG",
    "Opponent xG", "Opposition difficulty", "Expected to play", "Next opponent"]]
for column in ["Team xG", "Opponent xG"]:
    display_table[column] = display_table[column].map(
        lambda value: f"{value:.2f}" if pd.notna(value) else "—"
    )
display_table["Expected to play"] = display_table["Expected to play"].map(
    lambda value: f"{value:.0%}" if pd.notna(value) else "—"
)
print("Expected to play is recent appearance frequency; a WhoScored missing-player report sets it to 0%.")
print("xG columns are current-season averages for each team's last five matches, regardless of venue. Difficulty ranks the opponent's last-five-match xG within its league; higher xG is harder.")
if lookup_issues:
    print("Data coverage notes:")
    for issue in dict.fromkeys(lookup_issues):
        print(f"- {issue}")
display_table

[10/09/26 20:05:15] INFO     No custom team name replacements found. You can configure these in       ]8;id=8832270;file://C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=8832271;file://C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\soccerdata\_config.py#91\91]8;;\
                             C:\Users\User\soccerdata\config\teamname_replacements.json.                           

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=8832277;file://C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=8832278;file://C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\soccerdata\_config.py#189\189]8;;\
                             C:\Users\User\soccerdata\config\league_dict.json.                                     

[10/09/26 20:05:17] INFO     Saving cached data to C:\Users\User\soccerdata\data\ESPN                ]8;id=8832285;file://C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=8832286;file://C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\soccerdata\_common.py#250\250]8;;\

[2026-10-09 20:05:17] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll


                    INFO     Successfully loaded TLS library:                                      ]8;id=8832293;file://C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tls_requests\models\libraries.py\libraries.py]8;;\:]8;id=8832294;file://C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tls_requests\models\libraries.py#397\397]8;;\
                             C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.                 
                             3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\                 
                             tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll                              

...

                    INFO     [11/13] Retrieving game with id=1980985                               ]8;id=8833035;file://C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\soccerdata\whoscored.py\whoscored.py]8;;\:]8;id=8833036;file://C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\soccerdata\whoscored.py#525\525]8;;\

                    INFO     [12/13] Retrieving game with id=1980987                               ]8;id=8833041;file://C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\soccerdata\whoscored.py\whoscored.py]8;;\:]8;id=8833042;file://C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\soccerdata\whoscored.py#525\525]8;;\

                    INFO     [13/13] Retrieving game with id=1980993                               ]8;id=8833047;file://C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\soccerdata\whoscored.py\whoscored.py]8;;\:]8;id=8833048;file://C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\soccerdata\whoscored.py#525\525]8;;\

Expected to play is recent appearance frequency; a WhoScored missing-player report sets it to 0%.
xG columns are current-season averages for each team's last five matches, regardless of venue. Difficulty ranks the opponent's last-five-match xG within its league; higher xG is harder.
Data coverage notes:
- No current team/league mapping found for Matija Katic.


,Player,Team,Fixture date/time (Rome),Home/Away,Team xG,Opponent xG,Opposition difficulty,Expected to play,Next opponent
0,Vanja Milinković-Savić,Napoli,2026-10-10 20:45 CEST,Home,1.64,1.85,hard,50%,Frosinone
1,Alex Meret,Napoli,2026-10-10 20:45 CEST,Home,1.64,1.85,hard,50%,Frosinone
2,Luka Vuskovic,Brighton & Hove Albion,2026-10-10 16:00 CEST,Away,2.28,1.82,average,100%,Sunderland
3,Jeff Chabot,VfB Stuttgart,2026-10-10 15:30 CEST,Away,1.96,1.04,very easy,100%,SC Paderborn 07
4,Mario Hermoso,AS Roma,2026-10-11 12:30 CEST,Away,3.23,3.77,very hard,100%,Como
5,Lewis Dunk,Brighton & Hove Albion,2026-10-10 16:00 CEST,Away,2.28,1.82,average,0%,Sunderland
6,Jérémy Jacquet,Liverpool,2026-10-11 17:30 CEST,Home,1.54,1.85,hard,100%,Manchester City
7,Kristoffer Ajer,Brentford,2026-10-10 16:00 CEST,Away,1.20,0.68,very easy,100%,Aston Villa
8,Matija Katic,None,None,None,—,—,None,—,None
9,Scott McTominay,Napoli,2026-10-10 20:45 CEST,Home,1.64,1.85,hard,0%,Frosinone
